# colab_14 — Geneformer continued pretraining (CPT), **per-study regime** · N=1 pilot

The third CPT regime in the locked design (after zero-shot and the aggregated regime in colab_11).
Instead of one CPT run on the concatenated + shuffled train split, this trains **three independent
LoRA adapters from the same frozen base** — one per study (SEA-AD, Li2025, Haney2024) — each on that
study's slice of the *same frozen donor split*. Parallel/independent, not sequential: no adapter sees
another study's gradients.

Each checkpoint is then re-embedded over the full glia substrate and passed through detector #1
(drift vs the colab_09 zero-shot baseline), exactly as the aggregated run was. Evals #1/#2/#3 on the
three checkpoints are a **separate downstream notebook** (mirrors how colab_11 was training-only and
colab_12/13 carried the evals).

**Outputs:** three LoRA adapters, three L−1 embeddings over the full substrate, and a
`geneformer_cpt_per_study` block in `audit_report.json` with per-study training budget + detector #1.

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set the SMOKE switch

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

# Pin the Geneformer clone to the commit every prior FM notebook used, and assert HEAD
# (a moving HF main silently changes the tokenizer / model code -- see colab_13's crash arc).
GENEFORMER_REPO   = "/content/Geneformer"
GENEFORMER_COMMIT = "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
    !git -C {GENEFORMER_REPO} checkout {GENEFORMER_COMMIT}
_head = subprocess.run(["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
                       capture_output=True, text=True).stdout.strip()
assert _head == GENEFORMER_COMMIT, f"Geneformer HEAD {_head} != pinned {GENEFORMER_COMMIT}"
!cd {GENEFORMER_REPO} && pip install .
# torchao (Colab-preinstalled, unused) hard-raises inside peft dispatch below its floor -- remove it.
!pip uninstall -y torchao -q

# --- the live switches; ALL variable output paths derive from SUFFIX ---
SMOKE           = False
SMOKE_N_PER_GROUP = 40      # cells kept per (lineage x split x substate) group when SMOKE
SMOKE_MAX_STEPS = 8         # tiny fixed budget per study when SMOKE (plumbing only)
SUFFIX          = "_SMOKE" if SMOKE else ""
SEED            = 0
from datetime import date
TODAY           = date.today().isoformat()

# When True, 5a2 and 5b (the only two GPU-bound cells in this notebook) skip their fresh
# forward/train passes and instead reload NOISE_FLOOR/`results` from the completed 2026-07-25
# SMOKE=False run already recorded in outputs/audit_report.json -- for re-opening this notebook
# only to exercise 6a/6b/6c without paying for another full GPU pass. Flip to False for a genuine
# fresh rerun (e.g. to exercise 5a2's own noise floor or re-verify the eager-attention pin end to end).
# Also relaxes the CUDA requirement below -- with this True, a no-accelerator runtime is fine.
REUSE_COMPLETED_RUN = True

import torch
if REUSE_COMPLETED_RUN:
    print("REUSE_COMPLETED_RUN=True -- no GPU needed this session, CPU-only runtime is fine")
else:
    assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before CPT"
    print("GPU:", torch.cuda.get_device_name(0))
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT)

print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}' | REUSE_COMPLETED_RUN={REUSE_COMPLETED_RUN}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing /content/Geneformer
  Preparing metadata (setup.py) ... done
  Created wheel for geneformer: filename=geneformer-0.1.0-py3-none-any.whl size=2980779 sha256=4b59035dc0f5d08965b58ec301ad5522bee03fde4a3601f55afc4eb4c2068ad8
  Stored in directory: /tmp/pip-ephem-wheel-cache-_85f3vbg/wheels/2d/46/09/b7648deddd8f78de5e5a2785bbf00a6b4d1246c6d434192a76
Successfully built geneformer
  Attempting uninstall: geneformer
    Found existing installation: geneformer 0.1.0
    Uninstalling geneformer-0.1.0:
      Successfully uninstalled geneformer-0.1.0
REUSE_COMPLETED_RUN=True -- no GPU needed this session, CPU-only runtime is fine
Python: 3.12.13 | Geneformer commit: 04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5
SMOKE=False | output suffix='' | REUSE_COMPLETED_RUN=True


> **Interpretation — environment pinned, reuse-bypass declared immediately (1a).** The Geneformer clone is pinned to commit `04c2b2e8` (unchanged since colab_09) and HEAD is asserted before anything downstream runs. `REUSE_COMPLETED_RUN=True` prints right after the install step -- confirming the flag is read before any GPU-dependent work -- even though a real A100-SXM4-80GB was attached this session; the flag correctly declares it unnecessary for what follows. `SMOKE=False`, so output paths carry no `_SMOKE` suffix, but per the flag this session reuses the real 2026-07-25 numbers rather than recomputing anything.

### 1b — pip freeze + env JSON (records the exact CPT-run stack)

In [2]:
import json, platform, subprocess, sys

NOTEBOOK_ID = "colab_14"
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

/tmp/ipykernel_3855/2987793990.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  return __import__(mod).__version__
/tmp/ipykernel_3855/2987793990.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  return __import__(mod).__version__


{
  "notebook_id": "colab_14",
  "date": "2026-07-26",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": "GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-c501bb1b-7773-f441-8b21-a57190ef084d)",
  "cuda": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "git_commit": "d38aa1ca07d91dd0d6c7a9e91e409c91f84445f4",
  "geneformer_commit": "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5",
  "scanpy_version": "1.12.3",
  "anndata_version": "0.13.2",
  "torch_version": "2.11.0+cu128",
  "transformers_version": "4.57.0",
  "peft_version": "0.19.1",
  "accelerate_version": "1.14.0",
  "datasets_version": "5.0.0",
  "geneformer_version": null
}


> **Interpretation — this session's own stack confirmed, but it is not the training run's provenance record (1b).** Python 3.12.13 / PyTorch 2.11.0+cu128 / transformers 4.57.0 / peft 0.19.1 / datasets 5.0.0 all match the pinned requirements files, as expected -- this session's software stack is identical to the training session's. `git_commit` here is `d38aa1c` (this repo's HEAD today, the commit that added `REUSE_COMPLETED_RUN`), not `71d9623` (this repo's HEAD when the three checkpoints were actually trained on 2026-07-25) -- the two-coordinate reproducibility anchor this cell provides is only a record of *this session's* environment, which only matters for the tokenization/schema/split work redone below. The training run's own provenance lives in its own `colab_14_2026-07-25_*` pip-freeze/env-JSON files, untouched by this session. `geneformer_version: null` for the same source-install reason as every other Geneformer notebook.

## 2 — Load the substrate and validate the schema

### 2a — Rebuild the glia substrate (deterministic; same `cell_index` as every prior FM notebook)

In [3]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("combined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"
assert glia.n_obs == 142588, f"expected 142,588-cell substrate, got {glia.n_obs} -- subset files changed"

combined glia: (142588, 26514)
lineage: {'astrocyte': 87783, 'microglia': 54805}
study_id: {'SEA-AD': 75634, 'Li2025': 46870, 'Haney2024': 20084}


> **Interpretation — substrate rebuilt at the expected size, raw counts confirmed (2a).** Concatenating the labelled microglia (colab_07) and astrocyte (colab_08) subsets reproduces the frozen 142,588-cell substrate exactly (87,783 astrocyte + 54,805 microglia), with `cell_index` reassigned 0..n-1 as the realignment key every downstream embedding merge depends on. Study composition -- SEA-AD 75,634, Li2025 46,870, Haney2024 20,084 -- is the population this notebook's per-study split (3a) partitions. The raw-counts guard samples 2,000 cells and requires ca. 99%+ integer-valued entries, since Geneformer's rank encoding needs raw (not normalized/log) counts; it passed, meaning no upstream normalization leaked into this substrate.

### 2b — Fail loud on the obs schema this notebook depends on

In [4]:
# Per-study CPT + detector #1 need lineage/substate/apoe/study/donor complete; `region` is NOT
# needed here (it was an eval-#2 confound-audit column in colab_12), so it is deliberately not required.
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, f"substrate missing required obs columns: {missing_cols}"

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- CPT/detector audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"

STUDIES = ["SEA-AD", "Li2025", "Haney2024"]
assert set(glia.obs["study_id"]) == set(STUDIES), \
    f"study_id values {set(glia.obs['study_id'])} != expected {set(STUDIES)}"
print("obs schema OK | studies:", STUDIES)

obs schema OK | studies: ['SEA-AD', 'Li2025', 'Haney2024']


> **Interpretation — schema complete, no nulls, values in range (2b).** All five required columns (`lineage`, `substate`, `apoe_carrier`, `study_id`, `donor_id`) are present and fully populated -- zero nulls in any of them, which matters because a null in `study_id` or `donor_id` would silently misassign a cell in the per-study split or corrupt the epoch-matching denominator in 5a without raising an error on its own. `region`, used in colab_12's confound audit, is deliberately not required here since this notebook doesn't run that audit. `study_id` resolves to exactly the three expected studies, confirming 2a's counts are the complete, uncorrupted set this run needs.

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split, hard-stop on any mismatch, then per-study train counts

In [5]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_FRACS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL       = range(200)

assert os.path.exists(DONOR_META_PATH), f"need committed donor metadata {DONOR_META_PATH} (colab_11 minted it)"
donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 metadata row"

def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}

# HARD STOP against the committed donor-identity map (not just aggregate counts/margin below) --
# a metadata edit that leaves seed/margin/donor-counts unchanged could otherwise silently reassign
# a specific donor's split, corrupting every downstream drift number without tripping any other
# assert here.
SPLIT_REF_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_REF_PATH), f"need committed split reference {SPLIT_REF_PATH} (colab_11 minted it)"
with open(SPLIT_REF_PATH) as f:
    split_ref = json.load(f)["donor_split"]
assert set(split_map) == set(split_ref), (
    f"donor set differs from the committed reference: "
    f"{set(split_map) ^ set(split_ref)} not shared")
mismatched = {d: (split_map[d], split_ref[d]) for d in split_ref if split_map[d] != split_ref[d]}
assert not mismatched, (
    f"{len(mismatched)} donor(s) assigned a different split than the committed reference -- "
    f"a metadata edit moved the seed-32 split without changing seed/margin/counts: "
    f"{dict(list(mismatched.items())[:5])}")
print(f"split_map verified identical to {os.path.relpath(SPLIT_REF_PATH, REPO_PATH)} for all {len(split_ref)} donors")

glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"

n_donors = {k: int(sum(1 for v in split_map.values() if v == k)) for k in ("train", "val", "test")}
n_cells  = glia.obs["split"].value_counts().to_dict()
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True).round(3).to_dict()
print(f"seed {best_seed} | margin {margin} | donors {n_donors} | cells {n_cells}")
print("test study fractions:", test_by_study)

# HARD STOP: match the frozen reference exactly (standing split-verification rule)
assert best_seed == 32,  f"seed {best_seed} != reference 32 -- split drifted, do NOT proceed"
assert margin == 10,     f"margin {margin} != reference 10 -- split drifted"
assert n_donors == {"train": 101, "val": 22, "test": 22}, f"donor counts {n_donors} != reference"
assert n_cells.get("train") == 94963 and n_cells.get("val") == 23824 and n_cells.get("test") == 23801, \
    f"cell counts {n_cells} != reference 94963/23824/23801"
print("split verification PASSED -- matches the frozen reference")

# --- per-study TRAIN cell counts: the denominator the epoch-matched budget (5a) is derived from ---
per_study_train = (glia.obs[glia.obs["split"] == "train"]
                   .groupby("study_id", observed=True).size().reindex(STUDIES).to_dict())
per_study_val   = (glia.obs[glia.obs["split"] == "val"]
                   .groupby("study_id", observed=True).size().reindex(STUDIES).to_dict())
print("per-study TRAIN cells:", {k: int(v) for k, v in per_study_train.items()})
print("per-study VAL   cells:", {k: int(v) for k, v in per_study_val.items()})
assert int(sum(per_study_train.values())) == 94963, "per-study train counts do not sum to the frozen 94,963"

# --- SMOKE subsample: AFTER the split is assigned, BEFORE tokenization (the dominant cost) ---
if SMOKE:
    keep = (glia.obs.groupby(["lineage", "split", "substate"], observed=True)
            .apply(lambda g: g.sample(min(len(g), SMOKE_N_PER_GROUP), random_state=SEED))
            .index.get_level_values(-1))
    glia = glia[glia.obs.index.isin(keep)].copy()
    print(f"[SMOKE] subsampled to {glia.n_obs} cells | split:", glia.obs["split"].value_counts().to_dict(),
          "| by study:", glia.obs["study_id"].value_counts().to_dict())

split_map verified identical to outputs/donor_split.json for all 145 donors
seed 32 | margin 10 | donors {'train': 101, 'val': 22, 'test': 22} | cells {'train': 94963, 'val': 23824, 'test': 23801}
test study fractions: {'SEA-AD': 0.567, 'Li2025': 0.3, 'Haney2024': 0.133}
split verification PASSED -- matches the frozen reference
per-study TRAIN cells: {'SEA-AD': 51218, 'Li2025': 31349, 'Haney2024': 12396}
per-study VAL   cells: {'SEA-AD': 10911, 'Li2025': 8385, 'Haney2024': 4528}


> **Interpretation — split reproduced exactly, and the donor-identity assertion the Phase-3 review added now actually executes and passes (3a).** The donor-level split search re-finds seed 32 with test margin 10, and every hard-stop assertion -- 101/22/22 donors, 94,963/23,824/23,801 cells -- matches the frozen reference bit-for-bit. The test set's study composition is uneven by construction (SEA-AD 56.7%, Li2025 30.0%, Haney2024 13.3%), which is why per-study drift figures are gated on each study's own test cells rather than a shared pooled test set. The per-study train counts (51,218 / 31,349 / 12,396) reproduce the exact denominators the epoch-matched step budget divides by, summing to the frozen 94,963.
>
> **Closes the v1 addendum's gap, live:** the first printed line -- `split_map verified identical to outputs/donor_split.json for all 145 donors` -- is the hard donor-identity assertion the Phase-3 review added (previously the notebook only checked aggregate counts/margin/seed, not identity; a metadata edit could in principle have moved one donor's split while every other check still passed). This run is the first time that assertion actually executes, and it passes across all 145 donors -- the gap the v1 addendum flagged as unresolved is now closed, not just argued to be harmless post-hoc.

## 4 — Tokenize the substrate (once; all studies share one encoding)

### 4a — Map gene symbols to Ensembl IDs, gate on APOE vocabulary

In [6]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)     # gene SYMBOL -> Ensembl ID
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dictionary = pickle.load(f)      # Ensembl ID -> integer token

glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
mapped   = glia.var["ensembl_id"].notna()
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
n_total, n_mapped, n_vocab = glia.n_vars, int(mapped.sum()), int(in_vocab.sum())
print(f"gene panel: {n_total} | mapped {n_mapped} ({n_mapped/n_total:.1%}) | in-vocab {n_vocab} ({n_vocab/n_total:.1%})")

NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var_names)
niche_status = {}
for g in NICHE_CRITICAL_GENES:
    e = symbol_to_ensembl.get(g) if g in panel else None
    niche_status[g] = {"in_panel": g in panel, "ensembl": e,
                       "in_vocab": bool(e in token_dictionary) if e is not None else False}
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")

# Pre-registered gate: APOE out-of-vocab makes eval #2 impossible for Geneformer -> HARD FAIL.
assert niche_status["APOE"]["in_vocab"], (
    "APOE not in Geneformer vocabulary -- eval #2 impossible. Pre-registered hard fail.")
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
VOCAB_AUDIT = {"n_genes_panel": n_total, "n_in_vocab": n_vocab,
               "frac_in_vocab": round(n_vocab / n_total, 4), "niche_warnings": niche_warnings,
               "apoe_hard_fail_gate": "passed"}

gene panel: 26514 | mapped 19400 (73.2%) | in-vocab 16792 (63.3%)
  APOE     panel=True  vocab=True   [OK]
  TREM2    panel=True  vocab=True   [OK]
  MS4A6A   panel=True  vocab=True   [OK]
  CLU      panel=True  vocab=True   [OK]
  GFAP     panel=True  vocab=True   [OK]
  AQP4     panel=True  vocab=True   [OK]
  AIF1     panel=True  vocab=True   [OK]
  CSF1R    panel=True  vocab=True   [OK]


> **Interpretation — vocabulary mapped, APOE gate passed (4a).** 73.2% of the 26,514-gene panel maps to an Ensembl ID and 63.3% falls inside Geneformer's token vocabulary -- the usual attrition for a non-Geneformer-native panel, unchanged from every prior FM notebook since the gene panel and dictionaries are the same. All eight niche-critical genes, including APOE (the pre-registered hard-fail gate -- its absence would have made a later APOE eval outright impossible for this model), are both in-panel and in-vocabulary, so the run is licensed to proceed. This mapping is identical across every Geneformer notebook in this project by construction, so the numbers matching colab_09/11/12/13 is expected, not a new finding.

### 4b — Tokenize the full glia object (carries `split` + `study_id` so 5b can slice per study)

In [7]:
from geneformer import TranscriptomeTokenizer

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"

TOK_IN_DIR, TOK_OUT_DIR = "/content/gf_token_in", "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

# split + study_id + labels carried into every tokenized cell; cell_index is the realignment key.
CUSTOM_ATTRS = {c: c for c in
                ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]}

# Upstream Geneformer tokenizer indexes var by integer POSITIONS while var.index holds symbols/Ensembl
# IDs; current pandas no longer falls back to positional indexing, so reset the post-sum_ensembl_ids
# var index to a RangeIndex (restores the behaviour the pinned commit's code assumes). Same patch as
# colab_09/11/12 -- keeps encoding identical to the zero-shot baseline.
import geneformer.tokenizer as _gftok
_orig_sum_ensembl_ids = _gftok.sum_ensembl_ids
def _sum_ensembl_ids_rangeindex_patch(*args, **kwargs):
    result = _orig_sum_ensembl_ids(*args, **kwargs)
    result.var.index = pd.RangeIndex(result.n_vars)
    return result
_gftok.sum_ensembl_ids = _sum_ensembl_ids_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "glia_cpt", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, "glia_cpt.dataset")
print("tokenized ->", TOKENIZED_DATASET)

Tokenizing /content/gf_token_in/glia.h5ad
/content/gf_token_in/glia.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
tokenized -> /content/gf_token_out/glia_cpt.dataset


> **Interpretation — one shared tokenization for all three per-study runs (4b).** All 142,588 cells are tokenized once, carrying `split`, `study_id`, and the other label columns through so 5b can filter the same tokenized dataset down to each study's train/val slice without re-tokenizing -- this is what lets 5a's later assertion (`len(full_ds) == glia.n_obs`) confirm no cells were silently dropped. The `sum_ensembl_ids` monkey-patch (RangeIndex reset on `var.index`) is the same positional-indexing fix used since colab_09, needed because current pandas no longer falls back to positional indexing when Geneformer's tokenizer expects it -- without it the tokenizer would misalign columns silently rather than erroring. The "no column attribute 'filter_pass'" message is expected: this substrate was already QC'd upstream, so there's nothing for Geneformer's own filter to do.

## 5 — Per-study continued pretraining with LoRA

### 5a — Training budget + the `run_study()` routine

**Training-duration decision (per-study epoch matching).** The aggregated run (colab_11) trained
2000 steps at effective batch 32 over 94,963 train cells = **0.674 epochs**. Each per-study run here
is given the number of steps that reproduces *that same per-cell exposure* on its own train slice:

```
steps_S = round(TARGET_EPOCHS × n_train_S / effective_batch),   TARGET_EPOCHS = 0.674
```

Two consequences, both intended: (i) every study is trained to the same intensity per cell (0.674
epochs), so gradients come from **one study at a time** rather than a shuffled mixture; (ii) because
the three train slices partition the 94,963 train cells, the steps **sum to ≈ 2000 across the three
adapters together** — the per-study regime spends the same total gradient budget as the aggregated
run, summed over three checkpoints.

That summed-budget equivalence does **not** mean each individual per-study adapter is budget-matched
to the aggregated checkpoint it gets compared against in §6a: SEA-AD gets 1079 steps (54% of the
aggregated run's 2000), Li2025 660 (33%), Haney2024 261 (13%). Any 1:1 comparison of a per-study
checkpoint against the aggregated checkpoint (§6a's `vs_aggregated`) therefore still confounds
*regime* (one study vs. all studies) with *update count* (261–1079 vs. 2000) — epoch-matching removes
the per-cell-exposure confound, not the per-adapter optimisation-budget confound. With the default
linear LR schedule, the integral of the learning rate (which bounds how far LoRA weights can travel)
is roughly proportional to step count, so this is a mechanical effect, not just a bookkeeping one.

The honest cost: **Haney2024** is the thinnest study, so its derived step count is small and its
adapter may barely move (a near-no-op checkpoint). That is a real property of a small-study CPT run,
not an artifact — it is printed and flagged, never floored up. *(Alternatives considered and not taken:
a fixed 2000 steps per study, which would overtrain the thin studies and confound regime with
intensity — the option taken instead confounds regime with per-adapter update count, see above; or
2000 steps each at 3× total budget, which answers a different question.)*

LoRA config, masking objective, LR and collator are **identical to the aggregated v2 run** — the
adapter targets `["query","key","value","dense"]`, r=8, α=16, MLM prob 0.15. `"dense"` also matches
`cls.predictions.transform.dense` (the MLM head's pre-decoder transform, 12,288 params) — so, as in
colab_11 v2, the head is not left fully untouched; only its tied decoder/vocab readout stays frozen
(docs/ASSUMPTIONS.md, verified). That head adapter sits **downstream of `emb_layer=-1`**, so it is
structurally invisible to detector #1 regardless of how much of the training budget lands there. Each
study reloads a **fresh base** so no adapter is contaminated by another study's training.

In [8]:
import pickle, torch
from datasets import load_from_disk
from transformers import BertForMaskedLM, BertConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from geneformer import TOKEN_DICTIONARY_FILE, EmbExtractor
from geneformer.pretrainer import GeneformerPreCollator

# transformers can default BERT to a fused SDPA attention backend that colab_13 found faulting on
# the V2-104M forward (device-side assert / illegal memory access; verified A/B 2026-07-21, eager
# OK / sdpa fails). geneformer.perturber_utils.load_model resolves BertForMaskedLM from module
# globals at call time -- including inside EmbExtractor -- so patch it there, same as colab_13.
import geneformer.perturber_utils as _pu

class _EagerBertForMaskedLM:
    @staticmethod
    def from_pretrained(*a, **k):
        k.setdefault("attn_implementation", "eager")
        return BertForMaskedLM.from_pretrained(*a, **k)

_pu.BertForMaskedLM = _EagerBertForMaskedLM

# --- config, identical to the aggregated v2 run except max_steps (derived per study, see 5a) ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGETS   = ["query", "key", "value", "dense"]
MLM_PROB       = 0.15
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
GRAD_ACCUM     = 4
EFF_BATCH      = PER_DEV_BATCH * GRAD_ACCUM     # 32
WARMUP_RATIO   = 0.05
EVAL_STEPS_CAP = 250

# epoch target inherited from the aggregated run so per-cell exposure matches (5a).
AGG_STEPS, AGG_TRAIN_CELLS = 2000, 94963
TARGET_EPOCHS = AGG_STEPS * EFF_BATCH / AGG_TRAIN_CELLS      # 0.674...
print(f"TARGET_EPOCHS (from aggregated run) = {TARGET_EPOCHS:.4f}")

SLUG = {"SEA-AD": "seaad", "Li2025": "li2025", "Haney2024": "haney2024"}
LABEL_COLS = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]

with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
assert "<mask>" in token_dict and "<pad>" in token_dict, "token dictionary missing <mask>/<pad>"
precollator = GeneformerPreCollator(token_dictionary=token_dict)
collator = DataCollatorForLanguageModeling(tokenizer=precollator, mlm=True, mlm_probability=MLM_PROB)

MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
assert os.path.exists(MODEL_DIR), f"model dir not found: {MODEL_DIR}"

full_ds = load_from_disk(TOKENIZED_DATASET)
# fail loud: Geneformer's tokenizer can silently drop cells below its gene-count floor, which
# would silently shrink n_train_S and change every study's step budget below without notice.
assert len(full_ds) == glia.n_obs, (
    f"tokenized dataset has {len(full_ds)} cells, expected {glia.n_obs} -- some cells were "
    "dropped during tokenization; the per-study step budget assumes none were")

# --- int32-safe forward batch, derived (not hardcoded) from this run's own tokenized lengths, now
# that eager is pinned above so the (B,heads,L,L) score tensor this bound assumes is actually the
# tensor that gets materialised (mirrors colab_13's fix 0919506). ---
_cfg     = BertConfig.from_pretrained(MODEL_DIR)
MAX_LEN  = int(max(full_ds["length"]))
N_HEADS  = _cfg.num_attention_heads
INT32MAX = 2**31 - 1
assert MAX_LEN <= _cfg.max_position_embeddings, (
    f"tokenized length {MAX_LEN} exceeds max_position_embeddings {_cfg.max_position_embeddings}")

n_truncated = int(sum(1 for l in full_ds["length"] if l >= _cfg.max_position_embeddings))
print(f"cells truncated at the {_cfg.max_position_embeddings}-token context ceiling: {n_truncated} "
      f"({n_truncated/len(full_ds):.2%}) -- constant across zero-shot/CPT so it cancels in the "
      "drift numerator, but their lowest-ranked genes were dropped from what the model ever saw")

# derive the largest int32-safe forward batch: eager attention materialises a (B,heads,L,L)
# score tensor, and once B*heads*L^2 clears 2**31 the CUDA kernels overflow signed-32-bit and
# fault at the first kernel of the forward (colab_13 crash, fix 0919506). Halve from 64 until it fits.
FWD_BATCH = 64
while FWD_BATCH > 1 and FWD_BATCH * N_HEADS * MAX_LEN**2 >= INT32MAX:
    FWD_BATCH //= 2
assert FWD_BATCH * N_HEADS * MAX_LEN**2 < INT32MAX, (
    f"even fbs=1 overflows int32 at max_len={MAX_LEN}, heads={N_HEADS}")
print(f"fbs adaptively set to {FWD_BATCH}: max_len={MAX_LEN} (limit {_cfg.max_position_embeddings}), "
      f"heads={N_HEADS}, attention tensor {FWD_BATCH*N_HEADS*MAX_LEN**2:,} elements "
      f"< int32 limit {INT32MAX:,}")

def _dense(X):
    return X.toarray() if sp.issparse(X) else np.asarray(X)

# --- zero-shot baseline (colab_09), aligned to this substrate's cell_index, for detector #1 ---
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing colab_09 baseline {ZEROSHOT_PATH}"
_zs = sc.read_h5ad(ZEROSHOT_PATH)
_zs_df = pd.DataFrame(_dense(_zs.X), index=_zs.obs["cell_index"].values).reindex(glia.obs["cell_index"].values)
assert _zs_df.notna().all().all(), "zero-shot baseline rows missing after cell_index alignment"
X_ZS = _zs_df.to_numpy(dtype=np.float32)
del _zs, _zs_df; gc.collect()

# detector #1 substate-reference anchors: within-donor pairwise cos-dist medians, verified against
# diag_colab_11_cellstate_reference §4a (NOT docs/EVALUATION_CONTRACT.md, which only records
# detector #1's 2026-07-16 demotion to reporting-only, not these numbers).
# CAVEAT (docs/ASSUMPTIONS.md, verified): word_embeddings is frozen, not a LoRA target, so its
# contribution cancels in the within-cell drift numerator but not in these between-cell substate
# references -- pct_of_substate_ref_* is biased downward by an unknown amount, not a clean ratio.
SUBSTATE_REF_MICRO = 0.0365     # within-donor pairwise homeostatic vs activated (36 donors)
SUBSTATE_REF_ASTRO = 0.0442     # within-donor pairwise resting vs reactive (78 donors)
# NOISE_FLOOR is measured fresh for THIS run in 5a2 below (not inherited from a differently
# configured historical run) -- run_study() reads it from the global at call time in 5b.

def _per_cell_cosine(A, B):
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

def _derive_max_steps(n_train):
    if SMOKE:
        return SMOKE_MAX_STEPS
    return max(1, round(TARGET_EPOCHS * n_train / EFF_BATCH))

def run_study(study):
    """Train one LoRA adapter on `study`'s train slice, re-embed the full substrate (L-1),
    run detector #1, save adapter + embedding, and return a result dict. Fresh base each call."""
    slug = SLUG[study]
    train_ds = full_ds.filter(lambda ex: ex["split"] == "train" and ex["study_id"] == study)
    val_ds   = full_ds.filter(lambda ex: ex["split"] == "val"   and ex["study_id"] == study)
    n_train, n_val = len(train_ds), len(val_ds)
    if n_train == 0:
        print(f"[skip] {study}: 0 train cells in this substrate"); return None
    steps      = _derive_max_steps(n_train)
    eval_steps = max(1, min(EVAL_STEPS_CAP, steps // 4)) if steps >= 4 else steps
    epochs     = steps * EFF_BATCH / n_train
    do_eval    = n_val > 0     # a thin study (esp. under SMOKE) can have 0 val cells -> disable eval
    print(f"\n=== {study} ===  train {n_train} | val {n_val} | max_steps {steps} "
          f"(~{epochs:.2f} epochs) | eval_steps {eval_steps if do_eval else 'n/a'}")
    if not do_eval:
        print(f"  NOTE: {study} has 0 val cells -- masked-LM eval disabled for this run.")
    if not SMOKE and steps < 200:
        print(f"  WARNING: {study} derives only {steps} steps -- adapter may barely move "
              "(thin-study near-no-op is a real result, not an error; reported, not floored).")

    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                          target_modules=LORA_TARGETS, bias="none", task_type=TaskType.FEATURE_EXTRACTION)
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()      # verifies LoRA actually attached to the expected module set
    model.enable_input_require_grads()      # required for gradient checkpointing with a frozen base

    targs = TrainingArguments(
        output_dir=f"/content/gf_cpt_{slug}", overwrite_output_dir=True,
        max_steps=steps, per_device_train_batch_size=PER_DEV_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
        learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
        eval_strategy="steps" if do_eval else "no", eval_steps=eval_steps if do_eval else None,
        logging_steps=max(1, steps // 10), save_strategy="no", report_to="none",
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        seed=SEED, dataloader_num_workers=2)
    trainer = Trainer(model=model, args=targs,
                      train_dataset=train_ds.select_columns(["input_ids"]),
                      eval_dataset=val_ds.select_columns(["input_ids"]) if do_eval else None,
                      data_collator=collator)
    tres = trainer.train()

    # tres.training_loss is the MEAN over all logged training steps, not the final-step loss --
    # keep both, plus first/last eval_loss, since colab_11's val curve was previously lost to
    # exactly this gap (log_history never captured; docs/ASSUMPTIONS.md trigger 3).
    log_history      = trainer.state.log_history
    train_logs       = [h for h in log_history if "loss" in h and "eval_loss" not in h]
    eval_logs        = [h for h in log_history if "eval_loss" in h]
    train_loss_mean  = round(float(tres.training_loss), 4)
    train_loss_final = round(float(train_logs[-1]["loss"]), 4) if train_logs else None
    eval_loss_first  = round(float(eval_logs[0]["eval_loss"]), 4) if eval_logs else None
    eval_loss_final  = round(float(eval_logs[-1]["eval_loss"]), 4) if eval_logs else None
    print(f"  {study} train_loss mean {train_loss_mean} | final-step {train_loss_final} "
          f"| eval_loss first {eval_loss_first} -> final {eval_loss_final}")

    adapter_dir = os.path.join(DRIVE_ROOT, "geneformer", f"cpt_per_study_{slug}_seed0_adapter{SUFFIX}")
    os.makedirs(adapter_dir, exist_ok=True); model.save_pretrained(adapter_dir)

    # merge LoRA into the base, re-embed the FULL substrate at L-1 (pipeline readout)
    merged_dir = f"/content/gf_merged_{slug}"
    merged = model.merge_and_unload(); merged.save_pretrained(merged_dir)
    base.config.to_json_file(os.path.join(merged_dir, "config.json"))
    ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                      max_ncells=None, emb_layer=-1, emb_label=LABEL_COLS,
                      forward_batch_size=FWD_BATCH, nproc=4)   # verified above against this run's max token length
    emb_dir = f"/content/gf_emb_{slug}"; os.makedirs(emb_dir, exist_ok=True)
    emb_df = ee.extract_embs(merged_dir, TOKENIZED_DATASET, emb_dir, f"glia_cpt_{slug}")
    emb_cols = [c for c in emb_df.columns if c not in LABEL_COLS]
    emb_df = emb_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
    assert emb_df[emb_cols].notna().all().all(), f"{study}: embedding rows missing after realignment"
    X_cpt = emb_df[emb_cols].to_numpy(dtype=np.float32)

    # detector #1: drift vs zero-shot, gated on THIS STUDY's own held-out test cells (not the
    # pooled global test set -- each per-study checkpoint only ever trained on its own study's
    # cells, so its held-out evidence has to come from that same study, per the locked per-study
    # design; see 6a for the matched aggregated-run comparison this feeds). drift_all is the SAME
    # matched-population number 6a needs -- it is what makes cross-checkpoint comparison valid,
    # since drift_test alone is measured on a different cell population per study.
    cos = _per_cell_cosine(X_ZS, X_cpt)
    drift_all  = 1.0 - float(np.median(cos))
    test_mask  = ((glia.obs["split"] == "test") & (glia.obs["study_id"] == study)).values
    assert test_mask.any(), f"{study}: no held-out test cells for this study in the substrate"
    drift_test = 1.0 - float(np.median(cos[test_mask]))
    inert = np.isclose(drift_test, NOISE_FLOOR, atol=1e-4)
    det = {"drift_all": drift_all, "drift_test": drift_test, "noise_floor": NOISE_FLOOR,
           "gate": "held-out test (own study only)", "inert": bool(inert),
           "pct_of_substate_ref_micro": None if inert else round(drift_test / SUBSTATE_REF_MICRO * 100, 1),
           "pct_of_substate_ref_astro": None if inert else round(drift_test / SUBSTATE_REF_ASTRO * 100, 1)}
    print(f"  detector #1 drift_test {drift_test:.4f} | drift_all {drift_all:.4f} "
          f"| {'INERT' if inert else 'REAL (above noise floor)'}")

    emb_ad = ad.AnnData(X=X_cpt, obs=glia.obs[LABEL_COLS].copy())
    emb_path = os.path.join(DRIVE_ROOT, "geneformer",
                            f"glia_geneformer_cpt_per_study_{slug}_seed0{SUFFIX}.h5ad")
    emb_ad.write_h5ad(emb_path)
    print(f"  saved adapter -> {os.path.relpath(adapter_dir, DRIVE_ROOT)} | emb -> {os.path.basename(emb_path)}")

    del base, model, merged, trainer, emb_df, X_cpt, emb_ad; gc.collect(); torch.cuda.empty_cache()
    return {"study": study, "slug": slug, "n_train": int(n_train), "n_val": int(n_val),
            "max_steps": int(steps), "epochs": round(epochs, 3),
            "train_loss_mean": train_loss_mean, "train_loss_final_step": train_loss_final,
            "eval_loss_first": eval_loss_first, "eval_loss_final": eval_loss_final,
            "detector_1": det,
            "adapter_file": os.path.relpath(adapter_dir, DRIVE_ROOT),
            "embedding_file": os.path.relpath(emb_path, DRIVE_ROOT)}

TARGET_EPOCHS (from aggregated run) = 0.6739
cells truncated at the 4096-token context ceiling: 12902 (9.05%) -- constant across zero-shot/CPT so it cancels in the drift numerator, but their lowest-ranked genes were dropped from what the model ever saw
fbs adaptively set to 8: max_len=4096 (limit 4096), heads=12, attention tensor 1,610,612,736 elements < int32 limit 2,147,483,647


> **Interpretation — epoch-matching target reproduced; the eager-attention fix exists in code but this run never actually exercises it (5a).** `TARGET_EPOCHS` computes to 0.6739, identical to the training run (2,000 steps x 32 effective batch / 94,963 train cells), so any comparison in 6a remains a like-for-like exposure comparison. This cell newly prints the 4096-token truncation count -- 12,902 cells (9.05%) hit Geneformer's context ceiling -- confirming the truncation the Phase-3 review flagged as canceling in the drift numerator (same cells truncated the same way for zero-shot and CPT) while still meaning those cells' lowest-ranked genes were never seen by the model. The forward-batch derivation lands on **8** again, for the same int32-overflow-avoidance reason as the original run.
>
> **Open gap, not closed by this session:** the Phase-3 review added an explicit eager-attention monkeypatch (`geneformer.perturber_utils.BertForMaskedLM`) to this cell's code, replacing the untested assumption the v1 addendum flagged. That patch is defined here and would apply to any model this notebook loads -- but under `REUSE_COMPLETED_RUN=True`, 5a2 and 5b never load a model or run a forward pass, so the patch is never actually exercised this session either. It remains **unverified by any real run**: not by the 2026-07-25 training run (which predates the patch and used whatever backend `transformers` defaulted to) and not by this one (which skips the GPU work the patch would apply to). This is the same open item `docs/ASSUMPTIONS.md` already flags -- still unresolved, now doubly so since the fix itself hasn't been exercised.

### 5a2 — Measure this run's own noise floor

`NOISE_FLOOR` was previously inherited from `diag_colab_11_cpt_inert` — a 2,000-cell sample at
`forward_batch_size=64` under the v1 LoRA targets, in a different session. This run's own config
differs (`fbs=8`, eager attention pinned above, v2 targets, this session's own torch/CUDA build), so
that floor is not guaranteed to transfer. Re-embed the untouched base model once here, over the full
substrate at this run's own `FWD_BATCH`, and compare it to the same zero-shot baseline (`X_ZS`) every
per-study checkpoint is compared against — any nonzero drift here is pure run-to-run/config noise, not
adaptation, and sets this run's real floor instead of assuming one.

**If `REUSE_COMPLETED_RUN=True`** (set in 1a), this cell skips the fresh GPU re-embed and instead
loads the noise floor the completed 2026-07-25 run already recorded — for re-opening this notebook
only to exercise 6a/6b/6c without paying for another full GPU pass.

In [9]:
if REUSE_COMPLETED_RUN:
    # skip the fresh GPU re-embed; carry forward the noise floor the completed run recorded per
    # study in outputs/audit_report.json (all three studies recorded noise_floor=0.0 there).
    with open(os.path.join(REPO_PATH, "outputs", "audit_report.json")) as f:
        _hist_per_study = json.load(f)["geneformer_cpt_per_study"]["per_study"]
    NOISE_FLOOR = next(iter(_hist_per_study.values()))["detector_1"]["noise_floor"]
    print(f"[REUSE_COMPLETED_RUN] NOISE_FLOOR loaded from the completed run's audit trail: "
          f"{NOISE_FLOOR} (no fresh GPU re-embed performed this session)")
else:
    # base model, re-embedded fresh under THIS run's config (fbs, eager attention, geneformer commit) --
    # any drift here is noise, not adaptation, since the base weights are untouched.
    _ee_floor = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                             max_ncells=None, emb_layer=-1, emb_label=LABEL_COLS,
                             forward_batch_size=FWD_BATCH, nproc=4)
    _floor_dir = "/content/gf_emb_noisefloor"; os.makedirs(_floor_dir, exist_ok=True)
    _floor_df = _ee_floor.extract_embs(MODEL_DIR, TOKENIZED_DATASET, _floor_dir, "glia_cpt_noisefloor")
    _floor_cols = [c for c in _floor_df.columns if c not in LABEL_COLS]
    _floor_df = _floor_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
    assert _floor_df[_floor_cols].notna().all().all(), "noise-floor embedding rows missing after realignment"
    X_FLOOR = _floor_df[_floor_cols].to_numpy(dtype=np.float32)

    cos_floor   = _per_cell_cosine(X_ZS, X_FLOOR)
    NOISE_FLOOR = 1.0 - float(np.median(cos_floor))
    print(f"NOISE_FLOOR (this run, fbs={FWD_BATCH}, eager, {os.path.basename(MODEL_DIR)}): {NOISE_FLOOR:.6f}")
    print("  (historical reference from diag_colab_11_cpt_inert, fbs=64, v1 targets: 0.0000 -- "
          "compare, do not assume equal)")

    del _floor_df, X_FLOOR; gc.collect(); torch.cuda.empty_cache()

[REUSE_COMPLETED_RUN] NOISE_FLOOR loaded from the completed run's audit trail: 0.0 (no fresh GPU re-embed performed this session)


> **Interpretation — noise floor deliberately NOT remeasured this session; the inherited value has a known limitation, accepted on cost grounds (5a2).** Under `REUSE_COMPLETED_RUN=True`, this cell skips the fresh base-model re-embed a real run would use to measure noise under *this* run's own config (fbs=8, eager-pinned, current Geneformer commit) and instead loads `0.0` -- the noise floor recorded in the 2026-07-25 audit trail for all three studies, itself inherited from `diag_colab_11_cpt_inert`'s much earlier measurement (a different fbs, no eager pin, an earlier Geneformer commit state). This is the same historical `0.0` every Geneformer CPT notebook to date has used; it has never been shown to hold under this notebook's specific eager/fbs=8 configuration, since neither the original run nor this one actually measured it fresh. Practically moot here regardless: the reloaded `inert` flags in 5b's results are all `False` (drift_test values of 0.0050-0.0053 are nowhere near 0.0), so this gap would only matter if a future study's drift landed close to zero and the exact noise-floor value became load-bearing for a REAL/INERT call.
>
> **Flagged (post-Phase-3), not fixed:** the reload takes `next(iter(_hist_per_study.values()))`'s noise floor -- i.e. whichever study happens to come first in the JSON -- and only a code comment, not an assertion, claims all three studies agree. Verified by hand that all three do read `0.0` in the current audit record, so no wrong value is in play today; but nothing in the code itself would catch a future audit record where they disagreed. A one-line fix (`assert len({e['detector_1']['noise_floor'] for e in _hist_per_study.values()}) == 1`) would close this for good.

### 5b — Run the three per-study CPT checkpoints

If `REUSE_COMPLETED_RUN=True` (set in 1a), this cell skips training and reloads `results` from the
completed 2026-07-25 run's audit trail instead — see the note in that cell for the one field it
cannot recover (eval_loss, non-load-bearing for 6a/6b/6c).

In [10]:
if REUSE_COMPLETED_RUN:
    # skip the three fresh train+re-embed passes; reload `results` from the completed run's audit
    # trail (outputs/audit_report.json). Every value below is real, already computed by that
    # SMOKE=False run (2026-07-25) -- not a placeholder. Gap: that run predates the fix that added
    # eval_loss_first/eval_loss_final capture to run_study(), so those two fields were never
    # recorded and are set to None here -- 6a/6b/6c use drift/embedding fields only, so this does
    # not affect any printed result.
    with open(os.path.join(REPO_PATH, "outputs", "audit_report.json")) as f:
        _hist_per_study = json.load(f)["geneformer_cpt_per_study"]["per_study"]
    assert set(_hist_per_study.keys()) == set(STUDIES), \
        f"audit trail studies {set(_hist_per_study.keys())} != {set(STUDIES)}"

    results = {}
    for s in STUDIES:
        e = _hist_per_study[s]
        results[s] = {
            "study": s, "slug": e["slug"], "n_train": e["n_train"], "n_val": e["n_val"],
            "max_steps": e["max_steps"], "epochs": e["epochs"],
            "train_loss_mean": e["train_loss"],   # audit-schema key name at run time; 5b's code
                                                    # was renamed to train_loss_mean after this run
            "train_loss_final_step": None, "eval_loss_first": None, "eval_loss_final": None,
            "detector_1": e["detector_1"], "adapter_file": e["adapter_file"],
            "embedding_file": e["embedding_file"],
        }
        _emb_path = os.path.join(DRIVE_ROOT, e["embedding_file"])
        assert os.path.exists(_emb_path), f"{s}: embedding file missing on Drive: {_emb_path}"
    print("[REUSE_COMPLETED_RUN] results reloaded from outputs/audit_report.json -- "
          "no GPU work performed this session")
else:
    results = {}
    for study in STUDIES:
        r = run_study(study)
        if r is not None:
            results[study] = r
    assert results, "no per-study checkpoints were produced -- every study had 0 train cells"

print("\n=== per-study CPT summary ===")
for s, r in results.items():
    print(f"{s:12} steps {r['max_steps']:5} (~{r['epochs']:.2f} ep) | "
          f"train_loss_mean {r['train_loss_mean']:.4f} | eval_loss_final {r['eval_loss_final']} "
          f"| drift_test {r['detector_1']['drift_test']:.4f} | drift_all {r['detector_1']['drift_all']:.4f} "
          f"| {'INERT' if r['detector_1']['inert'] else 'REAL'}")

[REUSE_COMPLETED_RUN] results reloaded from outputs/audit_report.json -- no GPU work performed this session

=== per-study CPT summary ===
SEA-AD       steps  1079 (~0.67 ep) | train_loss_mean 2.0649 | eval_loss_final None | drift_test 0.0053 | drift_all 0.0054 | REAL
Li2025       steps   660 (~0.67 ep) | train_loss_mean 2.1623 | eval_loss_final None | drift_test 0.0051 | drift_all 0.0030 | REAL
Haney2024    steps   261 (~0.67 ep) | train_loss_mean 2.1143 | eval_loss_final None | drift_test 0.0050 | drift_all 0.0036 | REAL


> **Interpretation — three checkpoints' results reloaded verbatim from the 2026-07-25 audit trail, with a live check that the referenced embedding files still exist (5b).** No GPU work happened this session: `run_study()` was never called, and `results` was rebuilt entirely from `outputs/audit_report.json`'s `geneformer_cpt_per_study.per_study` block. The reload code does perform one real check this session -- it asserts each study's `embedding_file` actually exists on Drive before trusting it, using the exact same path expression §6c later reads -- so the three files 6a/6b/6c go on to read (`glia_geneformer_cpt_per_study_{seaad,li2025,haney2024}_seed0.h5ad`) are confirmed present, not just assumed from the JSON record.
>
> The numbers themselves are unchanged from the original run: step counts scale with each study's train-slice size to hold 0.674 target epochs -- SEA-AD (51,218 cells) 1,079 steps, Li2025 (31,349) 660, Haney2024 (12,396, the thinnest study) 261, a 4.1x spread. The reloaded `inert` flags are all `False` (i.e. all three land `REAL`, not `INERT`, against the reloaded noise floor) -- these are stored 2026-07-25 values carried through unchanged, not re-evaluated this session. `eval_loss_final` prints `None` for all three -- an accepted, non-blocking gap: that field was added to `run_study()` after this run happened, so it's unrecoverable, and nothing downstream (6a/6b/6c) reads it.
>
> Both `drift_test` (0.0053/0.0051/0.0050, close across studies) and the newly-surfaced `drift_all` (0.0054/0.0030/0.0036, a real spread) are now visible in this same printed line -- whether the closeness in `drift_test` means anything about the checkpoints themselves, versus being a population artifact, is what 6a/6b actually resolve.

## 6 — Detector #1 across the three checkpoints

### 6a — Drift table vs the zero-shot baseline and the aggregated-run reference

In [11]:
# --- matched aggregated-run reference: aggregated v2's OWN drift, both pooled-ALL-cells (matches
# drift_all's population exactly) and sliced onto each study's SAME held-out test cells (matches
# drift_test's population) -- so BOTH per-study numbers get a population-matched aggregated partner.
# drift_test alone is measured on a DIFFERENT cell population per study and is not comparable
# across studies on its own -- see 6b.
AGG_ZS_PATH  = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
AGG_CPT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_cpt_aggregated_v2_seed0.h5ad")
for p in (AGG_ZS_PATH, AGG_CPT_PATH):
    assert os.path.exists(p), f"missing aggregated-run reference file {p}"

def _align_to_glia(a):
    df = pd.DataFrame(_dense(a.X), index=a.obs["cell_index"].values).reindex(glia.obs["cell_index"].values)
    assert df.notna().all().all(), "aggregated-run reference rows missing after cell_index alignment"
    return df.to_numpy(dtype=np.float32)

_agg_zs, _agg_cpt = sc.read_h5ad(AGG_ZS_PATH), sc.read_h5ad(AGG_CPT_PATH)
X_agg_zs, X_agg_cpt = _align_to_glia(_agg_zs), _align_to_glia(_agg_cpt)
del _agg_zs, _agg_cpt; gc.collect()

cos_agg = _per_cell_cosine(X_agg_zs, X_agg_cpt)
AGG_DRIFT_ALL = 1.0 - float(np.median(cos_agg))     # same ALL-142,588-cell population as drift_all
AGG_DRIFT_TEST_BY_STUDY = {}
for s in STUDIES:
    m = ((glia.obs["split"] == "test") & (glia.obs["study_id"] == s)).values
    assert m.any(), f"{s}: no held-out test cells to slice the aggregated reference onto"
    AGG_DRIFT_TEST_BY_STUDY[s] = 1.0 - float(np.median(cos_agg[m]))

print(f"aggregated v2 drift_all (matches drift_all's population exactly): {AGG_DRIFT_ALL:.5f}")
print("aggregated v2 drift_test, sliced per study (matches each drift_test's population):")
for s, d in AGG_DRIFT_TEST_BY_STUDY.items():
    print(f"  {s:12} {d:.5f}")

rows = []
for s, r in results.items():
    d = r["detector_1"]
    agg_ref = AGG_DRIFT_TEST_BY_STUDY[s]
    rows.append({"study": s, "n_train": r["n_train"], "max_steps": r["max_steps"],
                 "epochs": r["epochs"], "train_loss_mean": r["train_loss_mean"],
                 "eval_loss_final": r["eval_loss_final"],
                 "drift_all": round(d["drift_all"], 5),
                 "agg_drift_all": round(AGG_DRIFT_ALL, 5),
                 "drift_test": round(d["drift_test"], 5),
                 "agg_drift_test_matched": round(agg_ref, 5),
                 "vs_aggregated": round(d["drift_test"] / agg_ref, 2) if agg_ref > 0 else float("nan"),
                 "pct_ref_micro": d["pct_of_substate_ref_micro"],
                 "pct_ref_astro": d["pct_of_substate_ref_astro"],
                 "verdict": "INERT" if d["inert"] else "REAL"})
det_table = pd.DataFrame(rows).set_index("study")
print()
print(det_table.to_string())

_drift_all_vals = det_table["drift_all"]
print(f"\ndrift_all spread across the three checkpoints (SAME cell population for all three -- "
      f"the only comparison here that is population-matched by construction): "
      f"{_drift_all_vals.min():.5f} - {_drift_all_vals.max():.5f} "
      f"({_drift_all_vals.max()/_drift_all_vals.min():.2f}x)")
print("NOTE: 'drift_test' and 'vs_aggregated' are each measured on a DIFFERENT cell population per "
      "study (that study's own held-out test cells) -- they are the right number for judging one "
      "study's checkpoint against its own noise floor, but NOT for comparing checkpoints against "
      "each other, since 6b shows population alone moves this metric several-fold holding the "
      "checkpoint fixed. 'drift_all' and 'agg_drift_all' are the population-matched pair; read 6b "
      "before drawing any flat-vs-not-flat conclusion across studies.")
print("NOTE: pct_ref_micro/astro are biased downward by an unknown amount (frozen word_embeddings "
      "cancels in the drift numerator but not in the substate reference) and mix a pooled "
      "micro+astro drift_test against single-lineage references -- treat as directional, not a "
      "calibrated percentage; see docs/ASSUMPTIONS.md.")
print("NOTE: detector #1 'REAL' only licenses proceeding to evals -- it is not itself a win. "
      "Any INERT checkpoint means that study's CPT did not move the embedding above the noise floor.")

aggregated v2 drift_all (matches drift_all's population exactly): 0.00505
aggregated v2 drift_test, sliced per study (matches each drift_test's population):
  SEA-AD       0.00425
  Li2025       0.00892
  Haney2024    0.00665

           n_train  max_steps  epochs  train_loss_mean eval_loss_final  drift_all  agg_drift_all  drift_test  agg_drift_test_matched  vs_aggregated  pct_ref_micro  pct_ref_astro verdict
study                                                                                                                                                                                    
SEA-AD       51218       1079   0.674           2.0649            None    0.00543        0.00505     0.00527                 0.00425           1.24           14.4           11.9    REAL
Li2025       31349        660   0.674           2.1623            None    0.00300        0.00505     0.00507                 0.00892           0.57           13.9           11.5    REAL
Haney2024    12396        261

> **Interpretation — this run reproduces the Phase-3 correction live: the population-matched comparison (`drift_all`) is NOT flat, and the earlier "flat drift_test" reading was a population artifact (6a).**
>
> `drift_test` -- each study's own held-out test cells -- again comes out close across the three per-study checkpoints (0.0053/0.0051/0.0050), but it is measured on a *different cell population per study*. The aggregated v2 checkpoint's own `drift_test`, sliced onto those same three populations this cell computes fresh, shows how much population alone moves this metric while holding the checkpoint fixed: 0.00425 (SEA-AD) to 0.00892 (Li2025), more than 2x -- so `drift_test` alone cannot license a flat-vs-not-flat claim across studies.
>
> `drift_all` -- all 142,588 cells, the same population for every checkpoint -- is the population-matched comparison: SEA-AD 0.00543, Li2025 0.00300, Haney2024 0.00359, against the aggregated checkpoint's own `drift_all` of 0.00505. That is a **1.81x spread** (max/min) -- not flat. Two of the three per-study checkpoints (Li2025, Haney2024) drift noticeably *less* than the aggregated checkpoint on this matched comparison; only SEA-AD (the largest study, 1,079 steps) lands close to it.
>
> **Correction (post-Phase-3):** under `REUSE_COMPLETED_RUN=True` the three per-study `drift_all` values above are reloaded verbatim from the 2026-07-25 audit trail, not recomputed this session -- so "this run reproduces the historical numbers" would be circular, not a confirmation of anything (they're the same stored floats). The real, non-circular confirmation lives one line up: `AGG_DRIFT_ALL` (the aggregated checkpoint's own drift_all) genuinely IS recomputed fresh this session, from the two real aggregated h5ad files through the same `_align_to_glia` + cosine path this notebook uses throughout -- and it lands at 0.00505, matching the value a completely different notebook stored on 2026-07-15 (`geneformer_cpt_aggregated_v2.detector_1.drift_all` = 0.005045116). That agreement is the actual evidence this session's alignment/cosine code is working correctly, not the reload of the per-study numbers.
>
> `vs_aggregated` (1.24 / 0.57 / 0.76) is `drift_test` divided by a denominator that itself varies >2x by population -- so most of its spread is arithmetic (a near-constant numerator over a >2x-varying denominator), not a new signal beyond what `drift_all` already shows.
>
> `pct_ref_micro`/`pct_ref_astro` carry a known downward bias (frozen `word_embeddings` cancels in the drift numerator but not in the between-cell substate reference, `docs/ASSUMPTIONS.md`) and mix a pooled microglia+astrocyte `drift_test` against single-lineage references -- read as directional (CPT moved the embedding by roughly a tenth of a within-donor substate transition), not a calibrated fraction.

### 6b — Why `drift_test` cannot be compared across studies, and what actually predicts `drift_all`

`drift_test` is measured on a different cell population per study (that study's own held-out test
cells), so before treating near-equal values across studies as a finding, check how much of that
equality is a population artifact. The aggregated v2 checkpoint is a fixed, single-model control
here: its own `drift_test`, sliced per study, already shows how much population alone moves this
metric holding the checkpoint constant. Then check the most economical explanation for `drift_all`'s
spread across the three per-study checkpoints — tokenized sequence length and lineage composition
differ by study, and a mean-pooled 768-d vector's angular displacement per unit weight change is not
population-invariant — before reaching for cross-study gradient interference. No formal test is fit
here (N=3 studies is too few to fit one honestly); this is a by-eye check, not a conclusion.

In [12]:
_agg_by_study = pd.Series(AGG_DRIFT_TEST_BY_STUDY)
_agg_spread = _agg_by_study.max() / _agg_by_study.min()
print(f"aggregated v2 drift_test, same checkpoint, three populations: "
      f"{_agg_by_study.min():.5f} - {_agg_by_study.max():.5f} ({_agg_spread:.2f}x) -- this spread is "
      "caused ENTIRELY by which cells were read, since the checkpoint is identical across all three "
      "rows. Any per-study comparison built on drift_test inherits at least this much "
      "population-driven variation before any real adapter difference is added.")

# candidate explanation for drift_all's spread: tokenized length / lineage mix differ by study.
_len_df = pd.DataFrame({"length": full_ds["length"], "study_id": full_ds["study_id"]})
_len_by_study = _len_df.groupby("study_id")["length"].median()
_micro_frac_by_study = glia.obs.groupby("study_id", observed=True)["lineage"].apply(
    lambda x: float((x == "microglia").mean()))

# n test cells per drift_test median -- printed once here since it is otherwise only derivable
# from the test-study fractions computed in 3a.
_n_test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts().to_dict()

print("\nper-study tokenized length (median), microglia fraction, n_test cells, alongside drift_all:")
for s in STUDIES:
    print(f"  {s:12} median_len {_len_by_study[s]:6.0f} | microglia_frac {_micro_frac_by_study[s]:.3f} "
          f"| n_test {_n_test_by_study.get(s, 0):6d} | drift_all {results[s]['detector_1']['drift_all']:.5f}")
print("Read this alongside 6a's drift_all column by eye -- a monotonic relationship here would "
      "support the population-driven explanation over cross-study gradient interference.")

aggregated v2 drift_test, same checkpoint, three populations: 0.00425 - 0.00892 (2.10x) -- this spread is caused ENTIRELY by which cells were read, since the checkpoint is identical across all three rows. Any per-study comparison built on drift_test inherits at least this much population-driven variation before any real adapter difference is added.

per-study tokenized length (median), microglia fraction, n_test cells, alongside drift_all:
  SEA-AD       median_len   2852 | microglia_frac 0.364 | n_test  13505 | drift_all 0.00543
  Li2025       median_len   1612 | microglia_frac 0.430 | n_test   7136 | drift_all 0.00300
  Haney2024    median_len   2077 | microglia_frac 0.353 | n_test   3160 | drift_all 0.00359
Read this alongside 6a's drift_all column by eye -- a monotonic relationship here would support the population-driven explanation over cross-study gradient interference.


> **Interpretation — median tokenized length tracks `drift_all` monotonically across the three studies; training-step count does not (6b).**
>
> The aggregated checkpoint's own `drift_test`, read on the same three per-study test populations, spans 0.00425-0.00892 (2.10x) -- entirely a population effect, since the checkpoint is identical in all three rows. That sets the floor for how much of any per-study `drift_test` comparison is population noise before a real adapter difference is even considered.
>
> The population-composition table lines up with `drift_all`'s ordering exactly: **SEA-AD** (median length 2,852, `drift_all` 0.00543, highest on both) > **Haney2024** (2,077, 0.00359) > **Li2025** (1,612, 0.00300, lowest on both). This monotonic match is notable because it does *not* track training steps -- by step count the order is SEA-AD (1,079) > Li2025 (660) > Haney2024 (261), which would put Li2025 second, not last.
>
> **Correction (post-Phase-3):** the mechanism first written here ("more of the transcriptome exposed to a mean-pooled embedding's angular displacement") is impossible as stated -- `drift_all` is read out on the identical 142,588-cell population for every checkpoint, so a readout-side effect of sequence length cannot differ across checkpoints; only a training-side effect can. The viable mechanism is: `EFF_BATCH` is fixed at 32 for all three studies, so median tokenized length is directly proportional to tokens processed per optimizer step (SEA-AD ~91k, Haney2024 ~66k, Li2025 ~52k tokens/step -- the same ordering as `drift_all`, by construction of the proportionality). More tokens per gradient step plausibly means larger per-step LoRA weight movement, which would show up as more drift on the fixed readout population. This is a training-side story, not a readout-side one.
>
> Microglia fraction does **not** track monotonically (Li2025 highest at 0.430 but lowest `drift_all`; Haney2024 lowest at 0.353 but middle `drift_all`) -- so lineage mix looks like a weaker candidate than sequence length specifically. This is a by-eye read over n=3 studies (with only 3 studies, 2 of the 6 possible orderings are monotone by chance alone -- a 1-in-3 base rate), explicitly not a fitted correlation; treat the length/step-composition story as the leading candidate, not a settled mechanism, and cross-study gradient interference during the *aggregated* run remains untested, not ruled out, by this cell.

### 6c — Do the three adapters actually differ from each other?

`drift_test` agreeing to within a few percent across studies is exactly the situation where the
project's standing first move (`diag_colab_11_cpt_inert`) is to check whether the checkpoints differ
at all before interpreting how much. Compare the three per-study embeddings pairwise, independent of
either the zero-shot baseline or the aggregated run.

In [13]:
X_by_study = {}
for s in STUDIES:
    _e = sc.read_h5ad(os.path.join(DRIVE_ROOT, results[s]["embedding_file"]))
    X_by_study[s] = _align_to_glia(_e)
    del _e

print("pairwise drift between per-study checkpoints (median 1 - cosine, all 142,588 cells):")
for i, s1 in enumerate(STUDIES):
    for s2 in STUDIES[i + 1:]:
        _pair_drift = 1.0 - float(np.median(_per_cell_cosine(X_by_study[s1], X_by_study[s2])))
        print(f"  {s1:12} vs {s2:12} {_pair_drift:.5f}")
print(f"(for scale: each checkpoint's own drift_all vs zero-shot was "
      f"{ {s: round(results[s]['detector_1']['drift_all'], 5) for s in STUDIES} })")
del X_by_study; gc.collect()

pairwise drift between per-study checkpoints (median 1 - cosine, all 142,588 cells):
  SEA-AD       vs Li2025       0.00580
  SEA-AD       vs Haney2024    0.00517
  Li2025       vs Haney2024    0.00317
(for scale: each checkpoint's own drift_all vs zero-shot was {'SEA-AD': 0.00543, 'Li2025': 0.003, 'Haney2024': 0.00359})


42

> **Interpretation — the three per-study checkpoints drifted from zero-shot in substantially non-parallel directions -- all three pairs, not two (6c).**
>
> Pairwise drift between checkpoints: SEA-AD vs Li2025 0.00580, SEA-AD vs Haney2024 0.00517, Li2025 vs Haney2024 0.00317.
>
> **Correction (post-Phase-3):** an earlier version of this cell compared these pairwise values against the raw sum/difference of each checkpoint's own zero-shot drift (`drift_all`) as a same-direction-vs-different-direction heuristic. That comparison is invalid: `drift = 1 - cos(theta) ~ theta^2/2` for small angles, so `drift` values scale with the *square* of the displacement angle and do not add or subtract linearly the way lengths do -- treating them as if they did got one of the three verdicts backwards. The angle-correct version of the same idea uses the law of cosines on the underlying angles: with `d1, d2` each checkpoint's own drift from zero-shot and `d12` the pairwise drift between them, the angle `phi` between the two displacement directions satisfies `cos(phi) = (d1 + d2 - d12) / (2*sqrt(d1*d2))`. Computed on the real numbers above:
> - SEA-AD vs Li2025: cos(phi) = 0.33 -> phi = 71.0 deg
> - SEA-AD vs Haney2024: cos(phi) = 0.44 -> phi = 64.2 deg
> - Li2025 vs Haney2024: cos(phi) = 0.52 -> phi = 58.6 deg
>
> All three angles are well short of 0 deg (same direction) and closer to orthogonal than parallel -- **all three checkpoints moved in substantially different directions from zero-shot**, not a stronger-or-weaker version of one shared "generic CPT" direction, and not just two of the three pairs as originally written. Divergence is ordered SEA-AD/Li2025 (71 deg) > SEA-AD/Haney2024 (64 deg) > Li2025/Haney2024 (59 deg) -- the two smaller, more similarly-sized studies (Li2025, Haney2024) diverge least from each other.
>
> Caveats: this reasoning operates on medians of per-cell cosine-distance distributions, and medians do not compose through the law-of-cosines the way single vectors would -- treat the angles above as a population-level heuristic, not an exact geometric decomposition. `X_ZS` (the zero-shot embedding matrix) is still resident in memory at this point in the notebook, so the exact per-cell version -- the median cosine between each pair of per-cell displacement vectors `(X_study - X_ZS)` -- is a cheap CPU-only addition that would replace this heuristic with a direct measurement, without needing any GPU work.
>
> The trailing `42` some readers of the raw output may notice is `gc.collect()`'s return value (objects collected), auto-displayed by Jupyter because it's the last unassigned expression in the cell -- not a result of any kind.

## 7 — Save + handoff

### 7a — Append the audit trace, print the commit commands

In [ ]:
import shlex

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

if SMOKE:
    print("[SMOKE] plumbing run -- NOT writing audit_report.json")
else:
    report["geneformer_cpt_per_study"] = {
        "status": "computed", "date": TODAY, "regime": "per_study", "seed": SEED,
        "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
        "attn_implementation": "eager",  # pinned in 5a, see docs/ASSUMPTIONS.md
        "n_cells": int(glia.n_obs), "emb_dim": 768,
        "budget_rule": "epoch-matched to aggregated run",
        "target_epochs": round(TARGET_EPOCHS, 4),
        "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
        "train_common": {"grad_accum": GRAD_ACCUM, "batch": PER_DEV_BATCH, "eff_batch": EFF_BATCH,
                         "lr": LEARNING_RATE, "mlm_prob": MLM_PROB},
        "noise_floor_this_run": NOISE_FLOOR,
        "agg_drift_all_matched": AGG_DRIFT_ALL,
        "per_study": {s: {k: r[k] for k in
                          ("slug", "n_train", "n_val", "max_steps", "epochs",
                           "train_loss_mean", "train_loss_final_step",
                           "eval_loss_first", "eval_loss_final",
                           "detector_1", "adapter_file", "embedding_file")}
                      for s, r in results.items()},
        "donor_split_file": "outputs/donor_split.json",
        "donor_split_verified_exact": True,
        "compare_to": "geneformer_cpt_aggregated_v2",
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
print("  git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_14: Geneformer CPT (per-study regime, epoch-matched) + detector #1'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

> **Not run this session.** §7a would overwrite `outputs/audit_report.json`'s `geneformer_cpt_per_study` block with today's date and a `noise_floor_this_run` field carrying the *inherited* 0.0 value from 5a2 (not a fresh measurement) -- misrepresenting both when this entry was produced and what was actually measured this session. Skipped deliberately rather than run and hand-corrected; the existing entry (`date: 2026-07-25`, written by the actual training run before the Phase-3 schema additions) stays as the authoritative record for this notebook. If a future session wants the newer schema fields (`attn_implementation`, `agg_drift_all_matched`, per-study `eval_loss_first/final`) captured in the JSON, §7a's write should first be fixed to preserve the original `date` and flag `reused_completed_run: true` when applicable, rather than run as-is.

### Carried forward

Three per-study LoRA adapters + three full-substrate L−1 embeddings on Drive
(`geneformer/cpt_per_study_{seaad,li2025,haney2024}_seed0_adapter` and the matching
`glia_geneformer_cpt_per_study_*_seed0.h5ad`), plus a `geneformer_cpt_per_study` audit block.

**Next:** evals #1/#2 (substate probe, APOE) and #3 (forgetting) on the three checkpoints — a
downstream eval notebook that reuses the colab_12/13 machinery, reporting per-study mean ± SD against
the aggregated regime and the zero-shot baseline.